# Spike V8: veri ve denetim (Adım 0)

**Ne yapar:** `configs/v2_v8.yaml`'daki bütün rol oturumlarını (ÖN, Day-0, AYAR, TEST, EK; Indy ve Loco) Zenodo'dan
indirir (md5 doğrulamalı), 4 ms önbelleğini ve görev durumu önbelleğini (`_task.pt`, ReFIT için) üretir ve
Drive'ım → `Spike_colab/cache/` altına yazar. Sonra **denetimi** (`scripts/audit_sessions.py`) koşar:
ÖN oturumlarının kanal düzeni Day-0 ile aynı mı, oran profili ne kadar kaymış. Rapor `Spike_colab/v8_audit.md`.

Dryad dosyaları (Chewie, Mihili, Han, Lando) elle konur: Drive'ım → `Spike_colab/dryad/` (en az
`Chewie_CO_20150630.mat`, 167 MB; https://doi.org/10.5061/dryad.xd2547dkt). Varsa denetime girer.

**Çalışma zamanı:** GPU gerekmez (CPU yeterli). Disk: ~25 GB geçici. Süre: indirme hızına göre ~30–60 dk.
Bağlantı koparsa yeniden **Tümünü çalıştır**: Drive'da önbelleği olan oturumlar yeniden üretilmez.

## 1. Ayarlar

In [ ]:
DRIVE_KLASOR = "/content/drive/MyDrive/Spike_colab"
CONFIG = "configs/v2_v8.yaml"
HAYVANLAR = ["indy", "loco"]
DENETIM_ICIN_MAT_TUT = True        # True: denetim tüm .mat'larla koşar (ρ(Day-0) için gerekli), sonda silinir

## 2. Drive ve kod

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, json, shutil, zipfile, pathlib, glob, subprocess, time
D = pathlib.Path(DRIVE_KLASOR)
assert (D / "spike_kod.zip").exists(), f"{D / 'spike_kod.zip'} yok: notebooks/README.md (V8) adım 1–3"
KOD = pathlib.Path("/content/Spike")
if KOD.exists():
    shutil.rmtree(KOD)
KOD.mkdir()
for f in ["spike_kod.zip", "spike_veri.zip"]:
    if (D / f).exists():
        with zipfile.ZipFile(D / f) as z:
            z.extractall(KOD)
os.chdir(KOD)
CACHE = KOD / "data/primate/cache"
CACHE.mkdir(parents=True, exist_ok=True)
DCACHE = D / "cache"                                   # v8_veri.ipynb'nin ürettiği önbellekler
DCACHE.mkdir(exist_ok=True)
for p in DCACHE.glob("*.pt"):
    if not (CACHE / p.name).exists():
        shutil.copy(p, CACHE / p.name)
surum = json.loads((KOD / "KOD_SURUMU.json").read_text(encoding="utf-8"))
print("kod:", surum["commit"], "(kirli)" if surum["dirty"] else "(temiz)", "| paket:", surum["paket_zamani"])
print("önbellek:", len(list(CACHE.glob("*.pt"))), "dosya")
ENV = {**os.environ, "PYTHONFAULTHANDLER": "1", "PYTHONIOENCODING": "utf-8"}

## 3. Bağımlılıklar

In [ ]:
!pip -q install pyyaml numpy h5py scipy neurobench==2.3.0

## 4. İndir ve önbellek üret

Her oturum: önbellek Drive'da yoksa indir (md5), üret, Drive'a kopyala.

In [ ]:
import sys
sys.path.insert(0, str(KOD))
from src.utils.config import load_config
from src.v2.data import check_roles, session_roles

cfg = load_config(CONFIG)
check_roles(cfg)
d = cfg["data"]
TAG = f"b{round(d['bin_width'] * 1000)}_s{round(d['stride'] * 1000)}_r{d['train_ratio']}"
MAT = KOD / d["dir"]
MAT.mkdir(parents=True, exist_ok=True)
adlar = [n for h in HAYVANLAR for n in session_roles(cfg, h).all()]
print(len(adlar), "oturum")


def py(*args):
    rc = subprocess.run(["python", *args], env=ENV).returncode
    if rc != 0:
        raise RuntimeError(f"başarısız: {args}")


for n in adlar:
    pt, task = f"{n}_{TAG}.pt", f"{n}_{TAG}_task.pt"
    hazir = (DCACHE / pt).exists() and (DCACHE / task).exists()
    if hazir and not DENETIM_ICIN_MAT_TUT:
        print("[atla]", n)
        continue
    t0 = time.time()
    if not (MAT / f"{n}.mat").exists():
        py("scripts/download_primate.py", "--config", CONFIG, "--names", n)
    if not hazir:
        py("scripts/build_cache.py", "--config", CONFIG, "--names", n)
        for f in (pt, task):
            shutil.copy(CACHE / f, DCACHE / f)
    print(f"[tamam] {n}  {(time.time() - t0) / 60:.1f} dk", flush=True)
    if not DENETIM_ICIN_MAT_TUT:
        (MAT / f"{n}.mat").unlink()

## 5. Denetim (Adım 0)

Kanal düzeni `FARKLI` çıkan ÖN oturumu varsa eğitime geçmeden önce Claude'a getir.

In [ ]:
DRY = KOD / "data/dryad"
DRY.mkdir(parents=True, exist_ok=True)
for p in ((D / "dryad").glob("*.mat") if (D / "dryad").exists() else []):
    shutil.copy(p, DRY / p.name)
py("scripts/audit_sessions.py", "--config", CONFIG, "--dryad-dir", str(DRY), "--out", "results/v8")
shutil.copy(KOD / "results/v8/audit.md", D / "v8_audit.md")
print((KOD / "results/v8/audit.md").read_text(encoding="utf-8"))

## 6. Temizlik ve özet

In [ ]:
for p in MAT.glob("*.mat"):
    p.unlink()
print("Drive önbelleği:", len(list(DCACHE.glob("*.pt"))), "dosya")
print("Denetim raporu:", D / "v8_audit.md", "→ bilgisayarına results/v8/audit.md olarak indir ve Claude'a göster.")